# TRAIN BRIGHTNESS EXTRACTION

In [ ]:
import cv2
import pandas as pd
import os
import sys
from tqdm import tqdm
import numpy as np

# --- Configuration ---
INPUT_CLEANED_CSV = "cleaned_train.csv"
OUTPUT_BRIGHTNESS_CSV = "extracted_brightness_features_train.csv"
VIDEO_ROOT_DIR = "barbados_traffic"

# Threshold: Average V-channel value (0-255). 
# Values above this are considered "Day". Adjust if needed.
BRIGHTNESS_THRESHOLD = 100 

# --- Core Function ---
def get_video_brightness_feature(video_path):
    """
    Calculates the average V-channel (brightness) across the first 20% of a video.
    This is faster than processing every frame.
    
    Args:
        video_path (str): Full path to the video file.
        
    Returns:
        tuple: (Average Brightness [0-255], Day/Night Label [str]) or None if error.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Process only the first 20% of the frames for a quick, representative average.
    # This keeps the script extremely fast.
    frames_to_sample = max(100, int(frame_count * 0.2)) 
    
    all_brightness_values = []
    
    for i in range(frames_to_sample):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert frame to HSV (Hue, Saturation, Value/Brightness)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        # Extract the V (Value) channel, which represents brightness/luminance
        V_channel = hsv[:, :, 2]
        
        # Calculate the average brightness of the frame
        mean_brightness = np.mean(V_channel)
        all_brightness_values.append(mean_brightness)

    cap.release()
    
    if not all_brightness_values:
        return None
        
    # Calculate the average brightness across the sampled frames
    avg_brightness = np.mean(all_brightness_values)
    
    # Assign Day or Night label based on the threshold
    day_night_label = "Day" if avg_brightness > BRIGHTNESS_THRESHOLD else "Night"
    
    return avg_brightness, day_night_label


# --- Main Execution ---
if __name__ == "__main__":
    
    # 1. Load Data
    try:
        df_master = pd.read_csv(INPUT_CLEANED_CSV)
        df_master['time_segment_id'] = df_master['time_segment_id'].astype(int)
        
        # Ensure 'videos' and 'view_label' columns exist
        if 'videos' not in df_master.columns or 'view_label' not in df_master.columns:
            print(f"CRITICAL ERROR: '{INPUT_CLEANED_CSV}' must contain 'videos' and 'view_label' columns.")
            sys.exit()
            
        print(f"Loaded {len(df_master)} video segments to process.")
        
    except FileNotFoundError:
        print(f"CRITICAL ERROR: '{INPUT_CLEANED_CSV}' not found.")
        sys.exit()
    except Exception as e:
        print(f"CRITICAL ERROR loading data: {e}. Exiting.")
        sys.exit()

    results = []
    total_segments = len(df_master)

    # 2. Iterate and Process Videos with TQDM Progress Bar
    for index, row in tqdm(df_master.iterrows(), total=total_segments, desc="💡 Extracting Brightness"):
        video_file = row['videos']
        time_segment_id = row['time_segment_id']
        view_label = row['view_label']
        
        # Construct the full path to the video
        video_path = os.path.join(VIDEO_ROOT_DIR, video_file)

        brightness_data = get_video_brightness_feature(video_path)
        
        if brightness_data:
            avg_brightness, day_night_label = brightness_data
            
            # Store results
            results.append({
                'time_segment_id': time_segment_id,
                'view_label': view_label,
                'videos': video_file,
                'Avg_Brightness_Value': round(avg_brightness, 2),
                'Day_Night_Label': day_night_label
            })
        else:
            tqdm.write(f"Warning: Could not open or process video file: {video_path}. Skipping.")
    
    # 3. Final Save
    if results:
        df_results = pd.DataFrame(results)
        
        # Create a unique ID for easy merging later
        df_results['ID'] = (
            df_results['time_segment_id'].astype(str) + '_' +
            df_results['view_label'].str.replace(' ', '_')
        )
        
        # Select and reorder final columns
        df_results = df_results[[
            'ID', 'time_segment_id', 'view_label', 'videos',
            'Avg_Brightness_Value', 'Day_Night_Label'
        ]]

        df_results.to_csv(OUTPUT_BRIGHTNESS_CSV, index=False)
        
        print("\n" + "═"*80)
        print(f"🎉 **TRAIN BRIGHTNESS EXTRACTION COMPLETE!**")
        print(f"✅ Extracted {len(df_results)} rows and saved to: **{OUTPUT_BRIGHTNESS_CSV}**")
        print("═"*80)
    else:
        print("\n❌ Extraction completed, but no brightness features were successfully extracted.")

Loaded 9014 video segments to process.


💡 Extracting Brightness:   0%|          | 32/9014 [00:18<1:11:13,  2.10it/s]

# Test Brightness Extraction

In [ ]:
import cv2
import pandas as pd
import os
import sys
from tqdm import tqdm
import numpy as np

# --- Configuration ---
INPUT_CLEANED_CSV = "TestInputSegments.csv"
OUTPUT_BRIGHTNESS_CSV = "extracted_brightness_features_test.csv"
VIDEO_ROOT_DIR = "barbados_traffic"

# Threshold: Average V-channel value (0-255). 
# Values above this are considered "Day". Adjust if needed.
BRIGHTNESS_THRESHOLD = 100 # its technically around 93-95 from the video analysis

# --- Core Function ---
def get_video_brightness_feature(video_path):
    """
    Calculates the average V-channel (brightness) across the first 20% of a video.
    This is faster than processing every frame.
    
    Args:
        video_path (str): Full path to the video file.
        
    Returns:
        tuple: (Average Brightness [0-255], Day/Night Label [str]) or None if error.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Process only the first 20% of the frames for a quick, representative average.
    # This keeps the script extremely fast.
    frames_to_sample = max(100, int(frame_count * 0.2)) 
    
    all_brightness_values = []
    
    for i in range(frames_to_sample):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert frame to HSV (Hue, Saturation, Value/Brightness)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        # Extract the V (Value) channel, which represents brightness/luminance
        V_channel = hsv[:, :, 2]
        
        # Calculate the average brightness of the frame
        mean_brightness = np.mean(V_channel)
        all_brightness_values.append(mean_brightness)

    cap.release()
    
    if not all_brightness_values:
        return None
        
    # Calculate the average brightness across the sampled frames
    avg_brightness = np.mean(all_brightness_values)
    
    # Assign Day or Night label based on the threshold
    day_night_label = "Day" if avg_brightness > BRIGHTNESS_THRESHOLD else "Night"
    
    return avg_brightness, day_night_label


# --- Main Execution ---
if __name__ == "__main__":
    
    # 1. Load Data
    try:
        df_master = pd.read_csv(INPUT_CLEANED_CSV)
        df_master['time_segment_id'] = df_master['time_segment_id'].astype(int)
        
        # Ensure 'videos' and 'view_label' columns exist
        if 'videos' not in df_master.columns or 'view_label' not in df_master.columns:
            print(f"CRITICAL ERROR: '{INPUT_CLEANED_CSV}' must contain 'videos' and 'view_label' columns.")
            sys.exit()
            
        print(f"Loaded {len(df_master)} video segments to process.")
        
    except FileNotFoundError:
        print(f"CRITICAL ERROR: '{INPUT_CLEANED_CSV}' not found.")
        sys.exit()
    except Exception as e:
        print(f"CRITICAL ERROR loading data: {e}. Exiting.")
        sys.exit()

    results = []
    total_segments = len(df_master)

    # 2. Iterate and Process Videos with TQDM Progress Bar
    for index, row in tqdm(df_master.iterrows(), total=total_segments, desc="💡 Extracting Brightness"):
        video_file = row['videos']
        time_segment_id = row['time_segment_id']
        view_label = row['view_label']
        
        # Construct the full path to the video
        video_path = os.path.join(VIDEO_ROOT_DIR, video_file)

        brightness_data = get_video_brightness_feature(video_path)
        
        if brightness_data:
            avg_brightness, day_night_label = brightness_data
            
            # Store results
            results.append({
                'time_segment_id': time_segment_id,
                'view_label': view_label,
                'videos': video_file,
                'Avg_Brightness_Value': round(avg_brightness, 2),
                'Day_Night_Label': day_night_label
            })
        else:
            tqdm.write(f"Warning: Could not open or process video file: {video_path}. Skipping.")
    
    # 3. Final Save
    if results:
        df_results = pd.DataFrame(results)
        
        # Create a unique ID for easy merging later
        df_results['ID'] = (
            df_results['time_segment_id'].astype(str) + '_' +
            df_results['view_label'].str.replace(' ', '_')
        )
        
        # Select and reorder final columns
        df_results = df_results[[
            'ID', 'time_segment_id', 'view_label', 'videos',
            'Avg_Brightness_Value', 'Day_Night_Label'
        ]]

        df_results.to_csv(OUTPUT_BRIGHTNESS_CSV, index=False)
        
        print("\n" + "═"*80)
        print(f"🎉 **TEST BRIGHTNESS EXTRACTION COMPLETE!**")
        print(f"✅ Extracted {len(df_results)} rows and saved to: **{OUTPUT_BRIGHTNESS_CSV}**")
        print("═"*80)
    else:
        print("\n❌ Extraction completed, but no brightness features were successfully extracted.")

Loaded 2640 video segments to process.


💡 Extracting Brightness:   5%|▍         | 127/2640 [01:12<18:46,  2.23it/s]

# DO NOT UNDERGO THE PAIN FOR WAITING LIKE WE DID IN THE ZIP YOU WILL FIND EXTRACTED FOLDER HAS ALL CSVS FROM THIS PROCESSES